# (1) AGMARKNET DATA COLLECTION

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pyarrow.parquet as pq

file_path = "/content/drive/MyDrive/Agmarknet_Master.parquet"

parquet_file = pq.ParquetFile(file_path)

print("Rows:", parquet_file.metadata.num_rows)
print("Columns:", parquet_file.schema.names)

Rows: 22995990
Columns: ['State', 'District', 'Market', 'Commodity Group', 'Commodity', 'Date', 'Arrival Quantity', 'Arrival Unit', 'Modal Price', 'Price Unit']


In [ ]:
#date inspect

import pyarrow.parquet as pq

table = pq.read_table(
    file_path,
    columns=["Date"]
)

date_df = table.to_pandas()

print(date_df.head())
print(date_df.tail())

print("\nData type:")
print(date_df.dtypes)

print("\nSample dates:")
print(date_df["Date"].sample(10, random_state=42))

         Date
0  01-11-2022
1  04-11-2022
2  05-11-2022
3  08-11-2022
4  11-11-2022
                Date
22995985  06-09-2025
22995986  07-09-2025
22995987  07-09-2025
22995988        None
22995989        None

Data type:
Date    object
dtype: object

Sample dates:
8524971     17-12-2023
14005052    26-10-2023
21075995    02-07-2024
8458490     15-06-2023
21690868    16-06-2024
7308811     27-01-2026
1615785     26-10-2024
18655414    12-06-2023
12749961    03-12-2025
10091380    01-10-2022
Name: Date, dtype: object


In [ ]:
#commodity check

table = pq.read_table(
    file_path,
    columns=["Commodity Group"]
)

group_df = table.to_pandas()

print(
    group_df["Commodity Group"]
    .value_counts()
)

Commodity Group
Vegetables     14821524
Fruits          2671935
Cereals         2144588
Pulses          1587527
Oil Seeds       1001922
Spices           499016
Fibre Crops      158059
Flowers           91609
Others            16196
-                  2354
Name: count, dtype: int64


In [ ]:
import pyarrow.parquet as pq

parquet_file = pq.ParquetFile(file_path)

print(parquet_file.schema_arrow)

State: string
District: string
Market: string
Commodity Group: string
Commodity: string
Date: string
Arrival Quantity: double
Arrival Unit: string
Modal Price: double
Price Unit: string
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 1460


In [ ]:
import pyarrow.parquet as pq
import pyarrow as pa
import pandas as pd

table = pq.read_table(
    file_path,
    filters=[
        ("Commodity Group", "==", "Vegetables")
    ]
)

print("Vegetable rows loaded:", table.num_rows)

# Converting to pandas
vegetable_df = table.to_pandas()

# Convert Date
vegetable_df["Date"] = pd.to_datetime(
    vegetable_df["Date"],
    format="%d-%m-%Y",
    errors="coerce"
)

# Apply exact date range
start_date = pd.Timestamp("2024-01-01")
end_date   = pd.Timestamp("2025-07-31")

vegetable_df = vegetable_df[
    (vegetable_df["Date"] >= start_date) &
    (vegetable_df["Date"] <= end_date)
].copy()

print("Final shape:", vegetable_df.shape)

display(vegetable_df.head())

Vegetable rows loaded: 14821524
Final shape: (5957033, 10)


,State,District,Market,Commodity Group,Commodity,Date,Arrival Quantity,Arrival Unit,Modal Price,Price Unit
0,Andaman and Nicobar,North and Middle Andaman,Mayabander Vegetable Market,Vegetables,Amaranthus,2024-11-04,0.05,Metric Tonnes,70.0,Rs./Quintal
1,Andaman and Nicobar,North and Middle Andaman,Mayabander Vegetable Market,Vegetables,Bhindi(Ladies Finger),2024-11-04,0.05,Metric Tonnes,60.0,Rs./Quintal
2,Andaman and Nicobar,North and Middle Andaman,Mayabander Vegetable Market,Vegetables,Brinjal,2024-11-04,0.05,Metric Tonnes,100.0,Rs./Quintal
3,Andaman and Nicobar,North and Middle Andaman,Mayabander Vegetable Market,Vegetables,Cowpea(Veg),2024-11-04,0.05,Metric Tonnes,70.0,Rs./Quintal
4,Andaman and Nicobar,North and Middle Andaman,Mayabander Vegetable Market,Vegetables,Cucumbar(Kheera),2024-11-04,0.05,Metric Tonnes,100.0,Rs./Quintal


In [ ]:
#verify

print("Date range:")
print(vegetable_df["Date"].min())
print(vegetable_df["Date"].max())

print("\nCommodity groups:")
print(vegetable_df["Commodity Group"].value_counts())

print("\nDistricts:", vegetable_df["District"].nunique())
print("Markets:", vegetable_df["Market"].nunique())
print("Vegetables:", vegetable_df["Commodity"].nunique())

Date range:
2024-01-01 00:00:00
2025-07-31 00:00:00

Commodity groups:
Commodity Group
Vegetables    5957033
Name: count, dtype: int64

Districts: 503
Markets: 1923
Vegetables: 82


In [ ]:
output_path = "/content/drive/MyDrive/Maharashtra_Vegetables_2024_2025.parquet"

vegetable_df.to_parquet(
    output_path,
    index=False
)

print("Saved successfully")
print(output_path)

Saved successfully
/content/drive/MyDrive/Maharashtra_Vegetables_2024_2025.parquet


In [ ]:
#only 2024
vegetable_2024 = vegetable_df[
    (vegetable_df["Date"] >= "2024-01-01") &
    (vegetable_df["Date"] <= "2024-12-31")
].copy()

print("Shape:", vegetable_2024.shape)
print("From:", vegetable_2024["Date"].min())
print("To:", vegetable_2024["Date"].max())

Shape: (3387384, 10)
From: 2024-01-01 00:00:00
To: 2024-12-31 00:00:00


In [ ]:
output_path = "/content/drive/MyDrive/Maharashtra_Vegetables_2024.parquet"

vegetable_2024.to_parquet(
    output_path,
    index=False
)

print("Saved:", output_path)

Saved: /content/drive/MyDrive/Maharashtra_Vegetables_2024.parquet


In [ ]:
import os

#ds info
print("DATASET INFO")
print("Rows:", vegetable_2024.shape[0])
print("Columns:", vegetable_2024.shape[1])

print("\nMEMORY")
print(f"{vegetable_2024.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

print("\nFILE INFO")
print("File:", output_path)
print(f"File size: {os.path.getsize(output_path) / (1024**2):.2f} MB")
print(f"File size: {os.path.getsize(output_path) / (1024**3):.2f} GB")

DATASET INFO
Rows: 3387384
Columns: 10

MEMORY
1471.97 MB

FILE INFO
File: /content/drive/MyDrive/Maharashtra_Vegetables_2024.parquet
File size: 12.97 MB
File size: 0.01 GB


In [ ]:
print("Date:", vegetable_2024["Date"].min(), "to", vegetable_2024["Date"].max())
print("Districts:", vegetable_2024["District"].nunique())
print("Markets:", vegetable_2024["Market"].nunique())
print("Vegetables:", vegetable_2024["Commodity"].nunique())

print("\nROWS/DIST")
print(vegetable_2024["District"].value_counts().head(20))

print("\nVEGETABLES")
print(sorted(vegetable_2024["Commodity"].dropna().unique()))

Date: 2024-01-01 00:00:00 to 2024-12-31 00:00:00
Districts: 495
Markets: 1809
Vegetables: 82

ROWS/DIST
District
Salem                 65074
Ernakulam             63544
Coimbatore            55086
Palakad               50441
Pune                  49650
Thiruvananthapuram    44970
Kottayam              43821
Thiruvannamalai       43404
Alappuzha             43357
Theni                 41223
Madurai               39396
Vellore               38307
Kozhikode(Calicut)    35587
Chengalpattu          34742
Namakkal              34349
Thiruchirappalli      34034
Malappuram            33608
Erode                 33562
Dharmapuri            32839
Thirssur              32453
Name: count, dtype: int64

VEGETABLES
['Alsandikai', 'Amaranthus', 'Amranthas Red', 'Ashgourd', 'Balekai', 'Banana - Green', 'Beetroot', 'Bhindi(Ladies Finger)', 'Bitter gourd', 'Bottle gourd', 'Brinjal', 'Cabbage', 'Capsicum', 'Carrot', 'Cauliflower', 'Chapparad Avare', 'Chilly Capsicum', 'Chow Chow', 'Cluster beans', 'Colac

In [ ]:
vegetable_2024["State"].value_counts()

,count
State,
Tamil Nadu,904437
Uttar Pradesh,563900
Keralam,445543
Punjab,247605
Maharashtra,174611
Haryana,170547
West Bengal,127089
Gujarat,109505
Odisha,95379


In [ ]:
maharashtra_veg_2024 = vegetable_2024[
    vegetable_2024["State"].str.strip().str.lower() == "maharashtra"
].copy()

print("Shape:", maharashtra_veg_2024.shape)
print("Districts:", maharashtra_veg_2024["District"].nunique())
print("Markets:", maharashtra_veg_2024["Market"].nunique())
print("Vegetables:", maharashtra_veg_2024["Commodity"].nunique())

Shape: (174611, 10)
Districts: 24
Markets: 124
Vegetables: 38


In [ ]:
# Maharashtra vegetables
maharashtra_veg = vegetable_df[
    vegetable_df["State"].str.strip().str.lower() == "maharashtra"
].copy()

print("Shape:", maharashtra_veg.shape)
print("Date:", maharashtra_veg["Date"].min(), "to", maharashtra_veg["Date"].max())
print("Districts:", maharashtra_veg["District"].nunique())
print("Markets:", maharashtra_veg["Market"].nunique())
print("Vegetables:", maharashtra_veg["Commodity"].nunique())

Shape: (287933, 10)
Date: 2024-01-01 00:00:00 to 2025-07-31 00:00:00
Districts: 24
Markets: 129
Vegetables: 38


In [ ]:
final_base_path = "/content/drive/MyDrive/Maharashtra_Vegetables_Jan2024_Jul2025.parquet"

maharashtra_veg.to_parquet(
    final_base_path,
    index=False
)

print("Saved successfully")

Saved successfully


In [ ]:
# into csv

csv_path = "/content/drive/MyDrive/Maharashtra_Vegetables_2024_Jul2025.csv"

maharashtra_veg.to_csv(
    csv_path,
    index=False
)

print("CSV saved successfully!")
print(csv_path)

CSV saved successfully!
/content/drive/MyDrive/Maharashtra_Vegetables_2024_Jul2025.csv


In [ ]:
import os

size_mb = os.path.getsize(csv_path) / (1024**2)

print(f"CSV file size: {size_mb:.2f} MB")

CSV file size: 29.36 MB


# CSV Check

In [ ]:
import pandas as pd

file_path = "/content/drive/MyDrive/Maharashtra_Vegetables_2024_Jul2025.csv"

df = pd.read_csv(file_path)

print("Shape:", df.shape)
df.head()

Shape: (287933, 10)


,State,District,Market,Commodity Group,Commodity,Date,Arrival Quantity,Arrival Unit,Modal Price,Price Unit
0,Maharashtra,Ahilyanagar,Ahilyanagar APMC,Vegetables,Onion,2024-11-04,4729.3,Metric Tonnes,3450.0,Rs./Quintal
1,Maharashtra,Ahilyanagar,Ahilyanagar APMC,Vegetables,Bhindi(Ladies Finger),2024-11-05,3.3,Metric Tonnes,2750.0,Rs./Quintal
2,Maharashtra,Ahilyanagar,Ahilyanagar APMC,Vegetables,Bitter gourd,2024-11-05,3.1,Metric Tonnes,2500.0,Rs./Quintal
3,Maharashtra,Ahilyanagar,Ahilyanagar APMC,Vegetables,Bottle gourd,2024-11-05,3.4,Metric Tonnes,1700.0,Rs./Quintal
4,Maharashtra,Ahilyanagar,Ahilyanagar APMC,Vegetables,Brinjal,2024-11-05,5.1,Metric Tonnes,3250.0,Rs./Quintal


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 287933 entries, 0 to 287932
Data columns (total 10 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   State             287933 non-null  object 
 1   District          287933 non-null  object 
 2   Market            287933 non-null  object 
 3   Commodity Group   287933 non-null  object 
 4   Commodity         287933 non-null  object 
 5   Date              287933 non-null  object 
 6   Arrival Quantity  287933 non-null  float64
 7   Arrival Unit      287933 non-null  object 
 8   Modal Price       287933 non-null  float64
 9   Price Unit        287933 non-null  object 
dtypes: float64(2), object(8)
memory usage: 22.0+ MB


In [ ]:
missing = df.isnull().sum()

print(missing)

State               0
District            0
Market              0
Commodity Group     0
Commodity           0
Date                0
Arrival Quantity    0
Arrival Unit        0
Modal Price         0
Price Unit          0
dtype: int64


In [ ]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [ ]:
print(df["Modal Price"].describe())
print()
print(df["Arrival Quantity"].describe())

count    287933.000000
mean       2844.842271
std        2754.652470
min           1.000000
25%        1300.000000
50%        2250.000000
75%        3500.000000
max      120000.000000
Name: Modal Price, dtype: float64

count    287933.000000
mean        213.755253
std         892.366026
min           0.100000
25%           1.000000
50%           3.600000
75%          24.800000
max        9995.000000
Name: Arrival Quantity, dtype: float64


In [ ]:
print("States:")
print(df["State"].unique())

print("\nNumber of districts:", df["District"].nunique())
print("Number of markets:", df["Market"].nunique())
print("Number of commodities:", df["Commodity"].nunique())

States:
['Maharashtra']

Number of districts: 24
Number of markets: 129
Number of commodities: 38


In [ ]:
import pandas as pd

file_path = "/content/drive/MyDrive/Maharashtra_Vegetables_2024_Jul2025.csv"

df = pd.read_csv(file_path)

print(df.shape)
print(df["Date"].head())
print(df["Date"].dtype)

(287933, 10)
0    2024-11-04
1    2024-11-05
2    2024-11-05
3    2024-11-05
4    2024-11-05
Name: Date, dtype: object
object


In [ ]:
# date format

df["Date"] = pd.to_datetime(df["Date"], errors="raise")

print("Date type:", df["Date"].dtype)
print("Min:", df["Date"].min())
print("Max:", df["Date"].max())
print("Missing dates:", df["Date"].isna().sum())

Date type: datetime64[ns]
Min: 2024-01-01 00:00:00
Max: 2025-07-31 00:00:00
Missing dates: 0


# (2) Adding Coordinates

market only master table for longitude and latitude to fetch weather data from NASA POWER

In [ ]:
markets = df[["State", "District", "Market"]].drop_duplicates()

print(markets.shape)
display(markets.head(20))

(129, 3)


,State,District,Market
0,Maharashtra,Ahilyanagar,Ahilyanagar APMC
39,Maharashtra,Ahilyanagar,Akole APMC
41,Maharashtra,Ahilyanagar,Jamkhed APMC
44,Maharashtra,Ahilyanagar,Karjat APMC
46,Maharashtra,Ahilyanagar,Kopargaon APMC
50,Maharashtra,Ahilyanagar,Newasa(Ghodegaon) APMC
51,Maharashtra,Ahilyanagar,Omchatinya Multi State Agro Purpose Co-Op Soci...
54,Maharashtra,Ahilyanagar,Parner APMC
57,Maharashtra,Ahilyanagar,Pathardi APMC
59,Maharashtra,Ahilyanagar,Rahata APMC


In [ ]:
markets = (
    df[["State", "District", "Market"]]
    .drop_duplicates()
    .sort_values(["District", "Market"])
    .reset_index(drop=True)
)

print("Total unique markets:", len(markets))

display(markets.head(20))

Total unique markets: 129


,State,District,Market
0,Maharashtra,Ahilyanagar,Ahilyanagar APMC
1,Maharashtra,Ahilyanagar,Akole APMC
2,Maharashtra,Ahilyanagar,Jamkhed APMC
3,Maharashtra,Ahilyanagar,Karjat APMC
4,Maharashtra,Ahilyanagar,Kopargaon APMC
5,Maharashtra,Ahilyanagar,Newasa(Ghodegaon) APMC
6,Maharashtra,Ahilyanagar,Omchatinya Multi State Agro Purpose Co-Op Soci...
7,Maharashtra,Ahilyanagar,Parner APMC
8,Maharashtra,Ahilyanagar,Pathardi APMC
9,Maharashtra,Ahilyanagar,Rahata APMC


In [ ]:
markets["Latitude"] = None
markets["Longitude"] = None

display(markets.head())

,State,District,Market,Latitude,Longitude
0,Maharashtra,Ahilyanagar,Ahilyanagar APMC,None,None
1,Maharashtra,Ahilyanagar,Akole APMC,None,None
2,Maharashtra,Ahilyanagar,Jamkhed APMC,None,None
3,Maharashtra,Ahilyanagar,Karjat APMC,None,None
4,Maharashtra,Ahilyanagar,Kopargaon APMC,None,None


In [ ]:
markets = (
    df[["State", "District", "Market"]]
    .drop_duplicates()
    .sort_values(["District", "Market"])
    .reset_index(drop=True)
)

markets["Latitude"] = None
markets["Longitude"] = None

print("Total markets:", len(markets))
display(markets)

Total markets: 129


,State,District,Market,Latitude,Longitude
0,Maharashtra,Ahilyanagar,Ahilyanagar APMC,None,None
1,Maharashtra,Ahilyanagar,Akole APMC,None,None
2,Maharashtra,Ahilyanagar,Jamkhed APMC,None,None
3,Maharashtra,Ahilyanagar,Karjat APMC,None,None
4,Maharashtra,Ahilyanagar,Kopargaon APMC,None,None
...,...,...,...,...,...
124,Maharashtra,Solapur,Kurdwadi(Modnimb) APMC,None,None
125,Maharashtra,Solapur,Laxmi-Sopan Agriculture Produce Marketing Co L...,None,None
126,Maharashtra,Solapur,Mangal Wedha APMC,None,None
127,Maharashtra,Solapur,Pandharpur APMC,None,None


In [ ]:
market_master_path = "/content/drive/MyDrive/Maharashtra_Market_Master.csv"

markets.to_csv(market_master_path, index=False)

print("Saved:", market_master_path)

Saved: /content/drive/MyDrive/Maharashtra_Market_Master.csv


In [ ]:
districts = (
    df[["State", "District"]]
    .drop_duplicates()
    .sort_values("District")
    .reset_index(drop=True)
)

print("Total districts:", len(districts))

display(districts)

Total districts: 24


,State,District
0,Maharashtra,Ahilyanagar
1,Maharashtra,Akola
2,Maharashtra,Amarawati
3,Maharashtra,Beed
4,Maharashtra,Buldhana
5,Maharashtra,Chandrapur
6,Maharashtra,Chattrapati Sambhajinagar
7,Maharashtra,Dharashiv
8,Maharashtra,Dhule
9,Maharashtra,Jalgaon


In [ ]:
import requests
import pandas as pd
import time

districts = (
    df[["State", "District"]]
    .drop_duplicates()
    .sort_values("District")
    .reset_index(drop=True)
)

district_search_names = {
    "Ahilyanagar": "Ahilyanagar",
    "Akola": "Akola",
    "Amarawati": "Amravati",
    "Beed": "Beed",
    "Buldhana": "Buldhana",
    "Chandrapur": "Chandrapur",
    "Chattrapati Sambhajinagar": "Chhatrapati Sambhajinagar",
    "Dharashiv": "Dharashiv",
    "Dhule": "Dhule",
    "Jalgaon": "Jalgaon",
    "Jalna": "Jalna",
    "Kolhapur": "Kolhapur",
    "Latur": "Latur",
    "Mumbai": "Mumbai",
    "Nagpur": "Nagpur",
    "Nandurbar": "Nandurbar",
    "Nashik": "Nashik",
    "Palghar": "Palghar",
    "Pune": "Pune",
    "Raigad": "Raigad",
    "Ratnagiri": "Ratnagiri",
    "Sangli": "Sangli",
    "Satara": "Satara",
    "Solapur": "Solapur"
}


def get_coordinates(place):

    url = "https://geocoding-api.open-meteo.com/v1/search"

    params = {
        "name": place,
        "count": 10,
        "language": "en",
        "format": "json",
        "countryCode": "IN"
    }

    try:
        response = requests.get(
            url,
            params=params,
            timeout=10
        )

        response.raise_for_status()

        results = response.json().get("results", [])

        # Only Maharashtra
        results = [
            r for r in results
            if r.get("admin1", "").lower() == "maharashtra"
        ]

        if not results:
            return None

        # result name closely matches our requested place
        for r in results:
            if r.get("name", "").lower() == place.lower():
                return r

        # Or else useing first Maharashtra result
        return results[0]

    except Exception as e:
        print("Error for", place, ":", e)
        return None


district_coordinates = []

for _, row in districts.iterrows():

    district = row["District"]
    search_name = district_search_names[district]

    result = get_coordinates(search_name)

    if result:

        district_coordinates.append({
            "State": row["State"],
            "District": district,
            "Geocoding_Name": search_name,
            "Latitude": result["latitude"],
            "Longitude": result["longitude"],
            "Geocoded_Name": result.get("name"),
            "Admin1": result.get("admin1"),
            "Admin2": result.get("admin2")
        })

        print(
            f" {district}: "
            f"{result['latitude']}, {result['longitude']}"
        )

    else:

        district_coordinates.append({
            "State": row["State"],
            "District": district,
            "Geocoding_Name": search_name,
            "Latitude": None,
            "Longitude": None,
            "Geocoded_Name": None,
            "Admin1": None,
            "Admin2": None
        })

        print(f" {district}: NOT FOUND")

    time.sleep(0.5)


district_master = pd.DataFrame(district_coordinates)

print("\nTotal districts:", len(district_master))
print(
    "Successfully geocoded:",
    district_master["Latitude"].notna().sum()
)
print(
    "Failed:",
    district_master["Latitude"].isna().sum()
)

display(district_master)

 Ahilyanagar: 19.09457, 74.73843
 Akola: 20.70957, 76.9981
 Amarawati: 20.93333, 77.75
 Beed: 18.98921, 75.75634
 Buldhana: NOT FOUND
 Chandrapur: 19.95076, 79.29523
 Chattrapati Sambhajinagar: NOT FOUND
 Dharashiv: 18.18158, 76.03889
 Dhule: 20.9013, 74.77737
 Jalgaon: 21.00292, 75.56602
 Jalna: 19.84102, 75.88636
 Kolhapur: 16.69563, 74.23167
 Latur: 18.39721, 76.56784
 Mumbai: 19.07283, 72.88261
 Nagpur: 21.14631, 79.08491
 Nandurbar: 21.36671, 74.24051
 Nashik: 19.99727, 73.79096
 Palghar: 19.69693, 72.76543
 Pune: 18.51957, 73.85535
 Raigad: NOT FOUND
 Ratnagiri: 16.99154, 73.31022
 Sangli: 16.85438, 74.56417
 Satara: 17.68589, 73.99333
 Solapur: 17.67152, 75.91044

Total districts: 24
Successfully geocoded: 21
Failed: 3


,State,District,Geocoding_Name,Latitude,Longitude,Geocoded_Name,Admin1,Admin2
0,Maharashtra,Ahilyanagar,Ahilyanagar,19.09457,74.73843,Ahilyanagar,Maharashtra,Ahmednagar
1,Maharashtra,Akola,Akola,20.70957,76.99810,Akola,Maharashtra,Akola district
2,Maharashtra,Amarawati,Amravati,20.93333,77.75000,Amravati,Maharashtra,Amravati
3,Maharashtra,Beed,Beed,18.98921,75.75634,Beed,Maharashtra,Beed District
4,Maharashtra,Buldhana,Buldhana,NaN,NaN,None,None,None
5,Maharashtra,Chandrapur,Chandrapur,19.95076,79.29523,Chandrapur,Maharashtra,Chandrapur
6,Maharashtra,Chattrapati Sambhajinagar,Chhatrapati Sambhajinagar,NaN,NaN,None,None,None
7,Maharashtra,Dharashiv,Dharashiv,18.18158,76.03889,Dharashiv,Maharashtra,Osmanabad District
8,Maharashtra,Dhule,Dhule,20.90130,74.77737,Dhule,Maharashtra,Dhule district
9,Maharashtra,Jalgaon,Jalgaon,21.00292,75.56602,Jalgaon,Maharashtra,Jalgaon


In [ ]:
manual_coordinates = {
    "Buldhana": (20.53278, 76.18278),
    "Chattrapati Sambhajinagar": (19.87634, 75.34509),
    "Raigad": (18.64000, 72.86000)
}

for district, (lat, lon) in manual_coordinates.items():

    district_master.loc[
        district_master["District"] == district,
        "Latitude"
    ] = lat

    district_master.loc[
        district_master["District"] == district,
        "Longitude"
    ] = lon

print("Manual coordinates added.")

Manual coordinates added.


In [ ]:
print(
    "Total districts:",
    len(district_master)
)

print(
    "With coordinates:",
    district_master["Latitude"].notna().sum()
)

print(
    "Still missing:",
    district_master["Latitude"].isna().sum()
)

display(
    district_master[
        ["District", "Latitude", "Longitude"]
    ]
)

Total districts: 24
With coordinates: 24
Still missing: 0


,District,Latitude,Longitude
0,Ahilyanagar,19.09457,74.73843
1,Akola,20.70957,76.99810
2,Amarawati,20.93333,77.75000
3,Beed,18.98921,75.75634
4,Buldhana,20.53278,76.18278
5,Chandrapur,19.95076,79.29523
6,Chattrapati Sambhajinagar,19.87634,75.34509
7,Dharashiv,18.18158,76.03889
8,Dhule,20.90130,74.77737
9,Jalgaon,21.00292,75.56602


In [ ]:
district_master_path = (
    "/content/drive/MyDrive/Maharashtra_District_Master.csv"
)

district_master.to_csv(
    district_master_path,
    index=False
)

print("Saved:", district_master_path)

Saved: /content/drive/MyDrive/Maharashtra_District_Master.csv


# (3) Weather Data

In [ ]:
#test
import requests
import pandas as pd

lat = 19.09457
lon = 74.73843

url = "https://power.larc.nasa.gov/api/temporal/daily/point"

params = {
    "parameters": "T2M,T2M_MIN,T2M_MAX,PRECTOTCORR,RH2M",
    "community": "AG",
    "longitude": lon,
    "latitude": lat,
    "start": "20240101",
    "end": "20240731",
    "format": "JSON"
}

response = requests.get(
    url,
    params=params,
    timeout=30
)

print("Status code:", response.status_code)

data = response.json()

print(data.keys())

Status code: 200
dict_keys(['type', 'geometry', 'properties', 'header', 'messages', 'parameters', 'times'])


In [ ]:
print(data["properties"]["parameter"].keys())

dict_keys(['T2M', 'T2M_MIN', 'T2M_MAX', 'PRECTOTCORR', 'RH2M'])


In [ ]:
import requests
import pandas as pd
import os
import time

# NASA POWER WEATHER DOWNLOAD

START_DATE = "20240101"
END_DATE   = "20250731"

OUTPUT_FILE = "/content/drive/MyDrive/Maharashtra_District_Weather_2024_2025.csv"

# Required NASA POWER parameters
PARAMETERS = "T2M,T2M_MIN,T2M_MAX,PRECTOTCORR,RH2M"

BASE_URL = "https://power.larc.nasa.gov/api/temporal/daily/point"


def get_weather(district, latitude, longitude):

    params = {
        "parameters": PARAMETERS,
        "community": "AG",
        "longitude": longitude,
        "latitude": latitude,
        "start": START_DATE,
        "end": END_DATE,
        "format": "JSON"
    }

    response = requests.get(
        BASE_URL,
        params=params,
        timeout=60
    )

    response.raise_for_status()

    data = response.json()

    weather = data["properties"]["parameter"]

    # Converting NASA date dictionary into dataframe
    weather_df = pd.DataFrame(weather)

    # NASA dates are dictionary keys
    weather_df.index.name = "Date"
    weather_df = weather_df.reset_index()

    # Converting date
    weather_df["Date"] = pd.to_datetime(
        weather_df["Date"],
        format="%Y%m%d"
    )

    # Add district and coordinates
    weather_df["District"] = district
    weather_df["Latitude"] = latitude
    weather_df["Longitude"] = longitude

    # Rename columns
    weather_df = weather_df.rename(columns={
        "T2M": "Temperature_C",
        "T2M_MIN": "Min_Temperature_C",
        "T2M_MAX": "Max_Temperature_C",
        "PRECTOTCORR": "Precipitation_mm",
        "RH2M": "Relative_Humidity_Percent"
    })

    # Reorder
    weather_df = weather_df[
        [
            "District",
            "Date",
            "Latitude",
            "Longitude",
            "Temperature_C",
            "Min_Temperature_C",
            "Max_Temperature_C",
            "Precipitation_mm",
            "Relative_Humidity_Percent"
        ]
    ]

    return weather_df

# LOAD EXISTING PROGRESS IF FILE ALREADY EXISTS

if os.path.exists(OUTPUT_FILE):

    weather_master = pd.read_csv(
        OUTPUT_FILE,
        parse_dates=["Date"]
    )

    completed_districts = set(
        weather_master["District"].unique()
    )

    print(
        f"Existing file found: "
        f"{len(completed_districts)} districts already completed."
    )

else:

    weather_master = pd.DataFrame()

    completed_districts = set()

    print("Starting weather downloading.")


# DOWNLOAD EACH DISTRICT

for _, row in district_master.iterrows():

    district = row["District"]
    latitude = row["Latitude"]
    longitude = row["Longitude"]

    # Skiping already completed districts
    if district in completed_districts:

        print(f" {district} already downloaded — skipping")

        continue

    print(
        f"\nDownloading: {district}"
    )

    try:

        weather_df = get_weather(
            district,
            latitude,
            longitude
        )

        print(
            f"  Records received: {len(weather_df)}"
        )

        # Adding to master
        weather_master = pd.concat(
            [
                weather_master,
                weather_df
            ],
            ignore_index=True
        )

        # Save
        weather_master.to_csv(
            OUTPUT_FILE,
            index=False
        )

        print(
            f"  Saved successfully"
        )

        #pause between API requests
        time.sleep(1)

    except Exception as e:

        print(
            f"  ✗ FAILED: {district}"
        )

        print(
            f"    Error: {e}"
        )


#FINAL SUMMARY

print("\n")
print("WEATHER DOWNLOAD COMPLETE")
print()

print(
    "Total records:",
    len(weather_master)
)

print(
    "Districts:",
    weather_master["District"].nunique()
)

print(
    "Date range:",
    weather_master["Date"].min(),
    "to",
    weather_master["Date"].max()
)

print(
    "Saved to:",
    OUTPUT_FILE
)

Starting weather downloading.

Downloading: Ahilyanagar
  Records received: 578
  Saved successfully

Downloading: Akola
  Records received: 578
  Saved successfully

Downloading: Amarawati
  Records received: 578
  Saved successfully

Downloading: Beed
  Records received: 578
  Saved successfully

Downloading: Buldhana
  Records received: 578
  Saved successfully

Downloading: Chandrapur
  Records received: 578
  Saved successfully

Downloading: Chattrapati Sambhajinagar
  Records received: 578
  Saved successfully

Downloading: Dharashiv
  Records received: 578
  Saved successfully

Downloading: Dhule
  Records received: 578
  Saved successfully

Downloading: Jalgaon
  Records received: 578
  Saved successfully

Downloading: Jalna
  Records received: 578
  Saved successfully

Downloading: Kolhapur
  Records received: 578
  Saved successfully

Downloading: Latur
  Records received: 578
  Saved successfully

Downloading: Mumbai
  Records received: 578
  Saved successfully

Downloading:

In [ ]:
# Load weather dataset
weather = pd.read_csv(
    "/content/drive/MyDrive/Maharashtra_District_Weather_2024_2025.csv",
    parse_dates=["Date"]
)

print("=" * 60)
print("WEATHER DATA QUALITY CHECK")
print("=" * 60)

# 1. Shape
print("\n1. Shape:")
print(weather.shape)

# 2. Districts
print("\n2. Number of districts:")
print(weather["District"].nunique())

# 3. Date range
print("\n3. Date range:")
print(weather["Date"].min(), "to", weather["Date"].max())

# 4. Records per district
print("\n4. Records per district:")
print(weather.groupby("District").size().describe())

# 5. Missing values
print("\n5. Missing values:")
print(weather.isna().sum())

# 6. Duplicate District + Date
print("\n6. Duplicate District + Date:")
duplicates = weather.duplicated(
    subset=["District", "Date"]
).sum()

print(duplicates)

# 7. NASA missing-value check
print("\n7. Possible NASA fill values (-999):")

weather_numeric = weather.select_dtypes(
    include="number"
)

print(
    (weather_numeric == -999).sum()
)

# 8. Numeric summary
print("\n8. Weather variable summary:")

display(
    weather[
        [
            "Temperature_C",
            "Min_Temperature_C",
            "Max_Temperature_C",
            "Precipitation_mm",
            "Relative_Humidity_Percent"
        ]
    ].describe()
)

# 9. Records per district
print("\n9. Records per district:")

display(
    weather.groupby("District")
    .size()
    .reset_index(name="Records")
    .sort_values("District")
)

# 10. First rows
print("\n10. Sample:")
display(weather.head(10))

WEATHER DATA QUALITY CHECK

1. Shape:
(13872, 9)

2. Number of districts:
24

3. Date range:
2024-01-01 00:00:00 to 2025-07-31 00:00:00

4. Records per district:
count     24.0
mean     578.0
std        0.0
min      578.0
25%      578.0
50%      578.0
75%      578.0
max      578.0
dtype: float64

5. Missing values:
District                     0
Date                         0
Latitude                     0
Longitude                    0
Temperature_C                0
Min_Temperature_C            0
Max_Temperature_C            0
Precipitation_mm             0
Relative_Humidity_Percent    0
dtype: int64

6. Duplicate District + Date:
0

7. Possible NASA fill values (-999):
Latitude                     0
Longitude                    0
Temperature_C                0
Min_Temperature_C            0
Max_Temperature_C            0
Precipitation_mm             0
Relative_Humidity_Percent    0
dtype: int64

8. Weather variable summary:


,Temperature_C,Min_Temperature_C,Max_Temperature_C,Precipitation_mm,Relative_Humidity_Percent
count,13872.000000,13872.000000,13872.000000,13872.000000,13872.000000
mean,26.827204,21.469324,33.129697,4.170604,59.248201
std,4.059040,4.581707,4.845522,10.437967,24.298150
min,14.120000,5.290000,22.840000,0.000000,6.090000
25%,24.270000,18.240000,29.250000,0.000000,35.860000
50%,26.560000,22.470000,31.830000,0.040000,64.610000
75%,29.450000,24.630000,36.912500,3.180000,81.332500
max,39.800000,33.590000,47.560000,253.460000,96.300000



9. Records per district:


,District,Records
0,Ahilyanagar,578
1,Akola,578
2,Amarawati,578
3,Beed,578
4,Buldhana,578
5,Chandrapur,578
6,Chattrapati Sambhajinagar,578
7,Dharashiv,578
8,Dhule,578
9,Jalgaon,578



10. Sample:


,District,Date,Latitude,Longitude,Temperature_C,Min_Temperature_C,Max_Temperature_C,Precipitation_mm,Relative_Humidity_Percent
0,Ahilyanagar,2024-01-01,19.09457,74.73843,22.13,14.18,30.76,0.00,49.43
1,Ahilyanagar,2024-01-02,19.09457,74.73843,22.70,14.50,30.93,0.00,48.88
2,Ahilyanagar,2024-01-03,19.09457,74.73843,23.31,15.62,30.30,0.00,49.49
3,Ahilyanagar,2024-01-04,19.09457,74.73843,23.88,17.58,31.56,0.00,44.49
4,Ahilyanagar,2024-01-05,19.09457,74.73843,22.90,16.66,29.60,0.12,46.09
5,Ahilyanagar,2024-01-06,19.09457,74.73843,23.44,17.59,30.07,0.53,53.69
6,Ahilyanagar,2024-01-07,19.09457,74.73843,24.15,16.98,31.13,0.00,45.88
7,Ahilyanagar,2024-01-08,19.09457,74.73843,24.36,17.36,31.69,0.00,43.18
8,Ahilyanagar,2024-01-09,19.09457,74.73843,22.62,19.12,28.06,3.58,50.81
9,Ahilyanagar,2024-01-10,19.09457,74.73843,23.02,16.63,29.71,1.11,57.56


In [ ]:
weather.to_csv(
    "/content/drive/MyDrive/Maharashtra_District_Weather_2024_2025.csv",
    index=False
)

print("Weather dataset checkpoint saved.")

Weather dataset checkpoint saved.


# (4) Soil Data

In [ ]:
!pip -q install rasterio requests

import requests
import rasterio
from rasterio.io import MemoryFile
import numpy as np
import pandas as pd
import time

In [ ]:
district_master = pd.read_csv(
    "/content/drive/MyDrive/Maharashtra_District_Master.csv"
)

print(district_master.shape)
display(
    district_master[
        ["District", "Latitude", "Longitude"]
    ]
)

(24, 8)


,District,Latitude,Longitude
0,Ahilyanagar,19.09457,74.73843
1,Akola,20.70957,76.99810
2,Amarawati,20.93333,77.75000
3,Beed,18.98921,75.75634
4,Buldhana,20.53278,76.18278
5,Chandrapur,19.95076,79.29523
6,Chattrapati Sambhajinagar,19.87634,75.34509
7,Dharashiv,18.18158,76.03889
8,Dhule,20.90130,74.77737
9,Jalgaon,21.00292,75.56602


In [ ]:
#soil property

soil_properties = {
    "phh2o": {
        "name": "pH",
        "conversion": 10
    },

    "soc": {
        "name": "Organic_Carbon_g_kg",
        "conversion": 10
    },

    "clay": {
        "name": "Clay_percent",
        "conversion": 10
    },

    "sand": {
        "name": "Sand_percent",
        "conversion": 10
    },

    "silt": {
        "name": "Silt_percent",
        "conversion": 10
    },

    "cec": {
        "name": "CEC_cmolc_kg",
        "conversion": 10
    },

    "nitrogen": {
        "name": "Nitrogen_g_kg",
        "conversion": 100
    }
}

depths = [
    "0-5cm",
    "5-15cm",
    "15-30cm"
]

print("Properties:", list(soil_properties.keys()))
print("Depths:", depths)

Properties: ['phh2o', 'soc', 'clay', 'sand', 'silt', 'cec', 'nitrogen']
Depths: ['0-5cm', '5-15cm', '15-30cm']


In [ ]:
# SoilGrids WCS function

WCS_URL = "https://maps.isric.org/mapserv"


def get_soil_raster(
    property_name,
    depth,
    latitude,
    longitude,
    radius=0.01
):

    coverage_id = (
        f"{property_name}_{depth}_mean"
    )

    params = [
        ("map", f"/map/{property_name}.map"),
        ("SERVICE", "WCS"),
        ("VERSION", "2.0.1"),
        ("REQUEST", "GetCoverage"),
        ("COVERAGEID", coverage_id),
        ("FORMAT", "GEOTIFF_INT16"),
        (
            "SUBSETTINGCRS",
            "http://www.opengis.net/def/crs/EPSG/0/4326"
        ),
        (
            "OUTPUTCRS",
            "http://www.opengis.net/def/crs/EPSG/0/4326"
        ),
        (
            "SUBSET",
            f"X({longitude-radius},{longitude+radius})"
        ),
        (
            "SUBSET",
            f"Y({latitude-radius},{latitude+radius})"
        )
    ]

    response = requests.get(
        WCS_URL,
        params=params,
        timeout=60
    )

    response.raise_for_status()

    return response.content

In [ ]:
def extract_soil_value(
    property_name,
    depth,
    latitude,
    longitude
):

    raster_bytes = get_soil_raster(
        property_name,
        depth,
        latitude,
        longitude
    )

    with MemoryFile(raster_bytes) as memfile:

        with memfile.open() as src:

            data = src.read(1).astype(float)


            valid = data[data > 0]

            if len(valid) == 0:
                return np.nan

            raw_mean = valid.mean()

            conversion = soil_properties[
                property_name
            ]["conversion"]

            converted_value = (
                raw_mean / conversion
            )

            return converted_value

In [ ]:
ahilyanagar = district_master[
    district_master["District"] == "Ahilyanagar"
].iloc[0]

lat = ahilyanagar["Latitude"]
lon = ahilyanagar["Longitude"]

print("Latitude:", lat)
print("Longitude:", lon)

Latitude: 19.09457
Longitude: 74.73843


In [ ]:
test_ph = extract_soil_value(
    "phh2o",
    "0-5cm",
    lat,
    lon
)

print("Ahilyanagar 0-5 cm pH:", test_ph)

Ahilyanagar 0-5 cm pH: 7.08


In [ ]:
test_clay = extract_soil_value(
    "clay",
    "0-5cm",
    lat,
    lon
)

print(
    "Ahilyanagar 0-5 cm clay (%):",
    test_clay
)

Ahilyanagar 0-5 cm clay (%): 42.766666666666666


In [ ]:
results = {}

for prop, info in soil_properties.items():
    value = extract_soil_value(
        prop,
        "0-5cm",
        lat,
        lon
    )
    results[info["name"]] = value
    print(f"{info['name']}: {value:.3f}")

print("\nTest complete.")

pH: 7.080
Organic_Carbon_g_kg: 12.773
Clay_percent: 42.767
Sand_percent: 28.673
Silt_percent: 28.573
CEC_cmolc_kg: 31.000
Nitrogen_g_kg: 1.277

Test complete.


In [ ]:
depth_test = {}

for depth in ["0-5cm", "5-15cm", "15-30cm"]:
    depth_test[depth] = {}

    for prop, info in soil_properties.items():
        value = extract_soil_value(
            prop,
            depth,
            lat,
            lon
        )

        depth_test[depth][info["name"]] = value

        print(f"{depth} | {info['name']}: {value:.3f}")

    print("-" * 60)

0-5cm | pH: 7.080
0-5cm | Organic_Carbon_g_kg: 12.773
0-5cm | Clay_percent: 42.767
0-5cm | Sand_percent: 28.673
0-5cm | Silt_percent: 28.573
0-5cm | CEC_cmolc_kg: 31.000
0-5cm | Nitrogen_g_kg: 1.277
------------------------------------------------------------
5-15cm | pH: 7.107
5-15cm | Organic_Carbon_g_kg: 9.907
5-15cm | Clay_percent: 41.667
5-15cm | Sand_percent: 29.527
5-15cm | Silt_percent: 28.827
5-15cm | CEC_cmolc_kg: 31.400
5-15cm | Nitrogen_g_kg: 1.154
------------------------------------------------------------
15-30cm | pH: 7.140
15-30cm | Organic_Carbon_g_kg: 7.320
15-30cm | Clay_percent: 42.347
15-30cm | Sand_percent: 27.767
15-30cm | Silt_percent: 29.893
15-30cm | CEC_cmolc_kg: 31.580
15-30cm | Nitrogen_g_kg: 0.930
------------------------------------------------------------


In [ ]:
soil_rows = []

for i, row in district_master.iterrows():

    district = row["District"]
    lat = row["Latitude"]
    lon = row["Longitude"]

    print(f"\n[{i+1}/24] Processing {district}...")

    result = {
        "District": district,
        "Latitude": lat,
        "Longitude": lon
    }

    for depth in ["0-5cm", "5-15cm", "15-30cm"]:

        for prop, info in soil_properties.items():

            try:
                value = extract_soil_value(
                    prop,
                    depth,
                    lat,
                    lon
                )

                column_name = f"{info['name']}_{depth}"
                result[column_name] = value

                print(
                    f"  {depth} | {info['name']}: {value:.3f}"
                )

            except Exception as e:
                column_name = f"{info['name']}_{depth}"
                result[column_name] = np.nan

                print(
                    f"  ERROR | {depth} | {info['name']} | {e}"
                )

    soil_rows.append(result)

    # Save progress after every district
    soil_temp = pd.DataFrame(soil_rows)
    soil_temp.to_csv(
        "/content/drive/MyDrive/Maharashtra_SoilGrids_Progress.csv",
        index=False
    )

print("\n===================================")
print("Soil extraction completed.")
print("===================================")


[1/24] Processing Ahilyanagar...
  0-5cm | pH: 7.080
  0-5cm | Organic_Carbon_g_kg: 12.773
  0-5cm | Clay_percent: 42.767
  0-5cm | Sand_percent: 28.673
  0-5cm | Silt_percent: 28.573
  0-5cm | CEC_cmolc_kg: 31.000
  0-5cm | Nitrogen_g_kg: 1.277
  5-15cm | pH: 7.107
  5-15cm | Organic_Carbon_g_kg: 9.907
  5-15cm | Clay_percent: 41.667
  5-15cm | Sand_percent: 29.527
  5-15cm | Silt_percent: 28.827
  5-15cm | CEC_cmolc_kg: 31.400
  5-15cm | Nitrogen_g_kg: 1.154
  15-30cm | pH: 7.140
  15-30cm | Organic_Carbon_g_kg: 7.320
  15-30cm | Clay_percent: 42.347
  15-30cm | Sand_percent: 27.767
  15-30cm | Silt_percent: 29.893
  15-30cm | CEC_cmolc_kg: 31.580
  15-30cm | Nitrogen_g_kg: 0.930

[2/24] Processing Akola...
  0-5cm | pH: nan
  0-5cm | Organic_Carbon_g_kg: nan
  0-5cm | Clay_percent: nan
  0-5cm | Sand_percent: nan
  0-5cm | Silt_percent: nan
  0-5cm | CEC_cmolc_kg: nan
  0-5cm | Nitrogen_g_kg: nan
  5-15cm | pH: nan
  5-15cm | Organic_Carbon_g_kg: nan
  5-15cm | Clay_percent: nan
  

In [ ]:
import pandas as pd
import numpy as np

# Load the completed SoilGrids extraction
soil = pd.read_csv(
    "/content/drive/MyDrive/Maharashtra_SoilGrids_Progress.csv"
)

# ---------------------------------------------------------
# 1. Create depth-weighted 0-30 cm values
# ---------------------------------------------------------

properties = {
    "pH": "pH",
    "Organic_Carbon_g_kg": "Organic_Carbon_g_kg",
    "Nitrogen_g_kg": "Nitrogen_g_kg",
    "Clay_percent": "Clay_percent",
    "CEC_cmolc_kg": "CEC_cmolc_kg"
}

depth_weights = {
    "0-5cm": 5,
    "5-15cm": 10,
    "15-30cm": 15
}

for prop in properties.values():

    cols = [
        f"{prop}_0-5cm",
        f"{prop}_5-15cm",
        f"{prop}_15-30cm"
    ]

    soil[f"{prop}_0_30"] = (
        soil[cols[0]] * 5 +
        soil[cols[1]] * 10 +
        soil[cols[2]] * 15
    ) / 30


# ---------------------------------------------------------
# 2. Keep only the 5 soil features we actually need
# ---------------------------------------------------------

final_soil = soil[
    [
        "District",
        "Latitude",
        "Longitude",
        "pH_0_30",
        "Organic_Carbon_g_kg_0_30",
        "Nitrogen_g_kg_0_30",
        "Clay_percent_0_30",
        "CEC_cmolc_kg_0_30"
    ]
].copy()

final_soil["Soil_Data_Method"] = np.where(
    final_soil["pH_0_30"].notna(),
    "SoilGrids",
    "Nearest successful district imputation"
)


# ---------------------------------------------------------
# 3. Impute the 9 missing districts
#    using nearest successful district
# ---------------------------------------------------------

soil_features = [
    "pH_0_30",
    "Organic_Carbon_g_kg_0_30",
    "Nitrogen_g_kg_0_30",
    "Clay_percent_0_30",
    "CEC_cmolc_kg_0_30"
]

successful = final_soil[
    final_soil["pH_0_30"].notna()
].copy()

missing = final_soil[
    final_soil["pH_0_30"].isna()
].copy()


def distance(lat1, lon1, lat2, lon2):
    return np.sqrt(
        (lat1 - lat2) ** 2 +
        (lon1 - lon2) ** 2
    )


for idx, row in missing.iterrows():

    distances = successful.apply(
        lambda x: distance(
            row["Latitude"],
            row["Longitude"],
            x["Latitude"],
            x["Longitude"]
        ),
        axis=1
    )

    nearest_idx = distances.idxmin()
    nearest = successful.loc[nearest_idx]

    for feature in soil_features:
        final_soil.loc[idx, feature] = nearest[feature]

    final_soil.loc[idx, "Soil_Data_Method"] = (
        f"Imputed from nearest district: {nearest['District']}"
    )


# ---------------------------------------------------------
# 4. Round values
# ---------------------------------------------------------

final_soil["pH_0_30"] = final_soil["pH_0_30"].round(3)
final_soil["Organic_Carbon_g_kg_0_30"] = final_soil[
    "Organic_Carbon_g_kg_0_30"
].round(3)

final_soil["Nitrogen_g_kg_0_30"] = final_soil[
    "Nitrogen_g_kg_0_30"
].round(3)

final_soil["Clay_percent_0_30"] = final_soil[
    "Clay_percent_0_30"
].round(3)

final_soil["CEC_cmolc_kg_0_30"] = final_soil[
    "CEC_cmolc_kg_0_30"
].round(3)


# ---------------------------------------------------------
# 5. Save final soil dataset
# ---------------------------------------------------------

output_path = (
    "/content/drive/MyDrive/"
    "Maharashtra_Soil_Master_0_30cm.csv"
)

final_soil.to_csv(output_path, index=False)

print("==============================================")
print("FINAL SOIL DATASET CREATED")
print("==============================================")
print("Shape:", final_soil.shape)
print("Saved:", output_path)

print("\nMissing values:")
print(final_soil.isna().sum())

display(final_soil)

FINAL SOIL DATASET CREATED
Shape: (24, 9)
Saved: /content/drive/MyDrive/Maharashtra_Soil_Master_0_30cm.csv

Missing values:
District                    0
Latitude                    0
Longitude                   0
pH_0_30                     0
Organic_Carbon_g_kg_0_30    0
Nitrogen_g_kg_0_30          0
Clay_percent_0_30           0
CEC_cmolc_kg_0_30           0
Soil_Data_Method            0
dtype: int64


,District,Latitude,Longitude,pH_0_30,Organic_Carbon_g_kg_0_30,Nitrogen_g_kg_0_30,Clay_percent_0_30,CEC_cmolc_kg_0_30,Soil_Data_Method
0,Ahilyanagar,19.09457,74.73843,7.119,9.091,1.063,42.190,31.423,SoilGrids
1,Akola,20.70957,76.99810,6.863,12.194,1.110,50.088,46.795,Imputed from nearest district: Buldhana
2,Amarawati,20.93333,77.75000,6.863,12.194,1.110,50.088,46.795,Imputed from nearest district: Buldhana
3,Beed,18.98921,75.75634,7.133,8.925,1.039,45.504,40.504,SoilGrids
4,Buldhana,20.53278,76.18278,6.863,12.194,1.110,50.088,46.795,SoilGrids
5,Chandrapur,19.95076,79.29523,7.120,9.287,0.996,34.830,34.863,SoilGrids
6,Chattrapati Sambhajinagar,19.87634,75.34509,7.288,7.556,0.863,44.860,43.335,Imputed from nearest district: Jalna
7,Dharashiv,18.18158,76.03889,7.210,9.121,1.025,40.776,41.202,SoilGrids
8,Dhule,20.90130,74.77737,7.172,7.106,0.827,42.339,39.672,SoilGrids
9,Jalgaon,21.00292,75.56602,7.200,7.342,0.955,38.550,37.558,SoilGrids


# (5) Crop Calendar

From ICAR

In [ ]:
# ============================================
# STEP 1: CREATE CROP CALENDAR TEMPLATE
# ============================================

import pandas as pd
import os

# Get the exact 38 vegetable names from our AGMARKNET dataset
vegetables = sorted(df["Commodity"].dropna().unique())

print("Number of vegetables:", len(vegetables))
print("\nVegetables:")
for i, crop in enumerate(vegetables, 1):
    print(f"{i:02d}. {crop}")

# Create crop calendar template
crop_calendar = pd.DataFrame({
    "Commodity": vegetables,
    "Crop_Duration_Days": pd.NA,
    "Suitable_Temp_Min_C": pd.NA,
    "Suitable_Temp_Max_C": pd.NA,
    "Suitable_pH_Min": pd.NA,
    "Suitable_pH_Max": pd.NA,
    "Suitable_Rainfall_Min_mm": pd.NA,
    "Suitable_Rainfall_Max_mm": pd.NA,
    "Typical_Sowing_Season": pd.NA,
    "Crop_Data_Source": pd.NA
})

# Save template
calendar_path = "/content/drive/MyDrive/Maharashtra_Vegetable_Crop_Calendar.csv"
crop_calendar.to_csv(calendar_path, index=False)

print("\nTemplate created:")
print(calendar_path)

print("\nShape:", crop_calendar.shape)
display(crop_calendar)

Number of vegetables: 38

Vegetables:
01. Banana - Green
02. Beetroot
03. Bhindi(Ladies Finger)
04. Bitter gourd
05. Bottle gourd
06. Brinjal
07. Cabbage
08. Carrot
09. Cauliflower
10. Chilly Capsicum
11. Chow Chow
12. Coriander(Leaves)
13. Cucumbar(Kheera)
14. Drumstick
15. Elephant Yam(Suran)/Amorphophallus
16. French Beans(Frasbean)
17. Garlic
18. Ginger(Green)
19. Green Chilli
20. Guar
21. Little gourd(Kundru)
22. Long Melon(Kakri)
23. Mango(Raw-Ripe)
24. Methi(Leaves)
25. Mint(Pudina)
26. Onion
27. Onion Green
28. Peas Wet
29. Pointed gourd(Parval)
30. Potato
31. Pumpkin
32. Raddish
33. Ridgeguard(Tori)
34. Snakeguard
35. Spinach
36. Sweet Potato
37. Sweet Pumpkin
38. Tomato

Template created:
/content/drive/MyDrive/Maharashtra_Vegetable_Crop_Calendar.csv

Shape: (38, 10)


,Commodity,Crop_Duration_Days,Suitable_Temp_Min_C,Suitable_Temp_Max_C,Suitable_pH_Min,Suitable_pH_Max,Suitable_Rainfall_Min_mm,Suitable_Rainfall_Max_mm,Typical_Sowing_Season,Crop_Data_Source
0,Banana - Green,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,Beetroot,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,Bhindi(Ladies Finger),<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,Bitter gourd,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,Bottle gourd,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
5,Brinjal,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
6,Cabbage,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
7,Carrot,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
8,Cauliflower,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
9,Chilly Capsicum,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [ ]:
# ============================================================
# STEP 2: POPULATE CROP DURATION + TYPICAL SOWING SEASON
# ============================================================

crop_info = {

    "Banana - Green": {
        "days": 300,
        "season": "Year-round"
    },

    "Beetroot": {
        "days": 70,
        "season": "Rabi / Winter"
    },

    "Bhindi(Ladies Finger)": {
        "days": 55,
        "season": "Summer / Kharif"
    },

    "Bitter gourd": {
        "days": 60,
        "season": "Summer / Kharif"
    },

    "Bottle gourd": {
        "days": 60,
        "season": "Summer / Kharif"
    },

    "Brinjal": {
        "days": 90,
        "season": "Kharif / Rabi / Summer"
    },

    "Cabbage": {
        "days": 90,
        "season": "Rabi / Winter"
    },

    "Carrot": {
        "days": 75,
        "season": "Rabi / Winter"
    },

    "Cauliflower": {
        "days": 80,
        "season": "Rabi / Winter"
    },

    "Chilly Capsicum": {
        "days": 90,
        "season": "Kharif / Rabi"
    },

    "Chow Chow": {
        "days": 120,
        "season": "Kharif / Mild season"
    },

    "Coriander(Leaves)": {
        "days": 40,
        "season": "Rabi / Winter"
    },

    "Cucumbar(Kheera)": {
        "days": 50,
        "season": "Summer / Kharif"
    },

    "Drumstick": {
        "days": 180,
        "season": "Year-round"
    },

    "Elephant Yam(Suran)/Amorphophallus": {
        "days": 240,
        "season": "Kharif"
    },

    "French Beans(Frasbean)": {
        "days": 70,
        "season": "Rabi / Summer"
    },

    "Garlic": {
        "days": 150,
        "season": "Rabi / Winter"
    },

    "Ginger(Green)": {
        "days": 210,
        "season": "Kharif"
    },

    "Green Chilli": {
        "days": 90,
        "season": "Kharif / Rabi"
    },

    "Guar": {
        "days": 70,
        "season": "Kharif / Summer"
    },

    "Little gourd(Kundru)": {
        "days": 120,
        "season": "Kharif / Year-round"
    },

    "Long Melon(Kakri)": {
        "days": 55,
        "season": "Summer"
    },

    "Mango(Raw-Ripe)": {
        "days": 150,
        "season": "Year-round orchard crop"
    },

    "Methi(Leaves)": {
        "days": 35,
        "season": "Rabi / Winter"
    },

    "Mint(Pudina)": {
        "days": 60,
        "season": "Year-round"
    },

    "Onion": {
        "days": 120,
        "season": "Kharif / Rabi"
    },

    "Onion Green": {
        "days": 60,
        "season": "Rabi / Winter"
    },

    "Peas Wet": {
        "days": 70,
        "season": "Rabi / Winter"
    },

    "Pointed gourd(Parval)": {
        "days": 120,
        "season": "Kharif / Summer"
    },

    "Potato": {
        "days": 100,
        "season": "Rabi / Winter"
    },

    "Pumpkin": {
        "days": 90,
        "season": "Summer / Kharif"
    },

    "Raddish": {
        "days": 45,
        "season": "Rabi / Winter"
    },

    "Ridgeguard(Tori)": {
        "days": 45,
        "season": "Summer / Kharif"
    },

    "Snakeguard": {
        "days": 60,
        "season": "Summer / Kharif"
    },

    "Spinach": {
        "days": 35,
        "season": "Rabi / Winter"
    },

    "Sweet Potato": {
        "days": 120,
        "season": "Kharif"
    },

    "Sweet Pumpkin": {
        "days": 90,
        "season": "Summer / Kharif"
    },

    "Tomato": {
        "days": 120,
        "season": "Kharif / Rabi / Summer"
    }
}


# Populate only the fields we currently need
for crop, info in crop_info.items():
    crop_calendar.loc[
        crop_calendar["Commodity"] == crop,
        "Crop_Duration_Days"
    ] = info["days"]

    crop_calendar.loc[
        crop_calendar["Commodity"] == crop,
        "Typical_Sowing_Season"
    ] = info["season"]


# Add definition of duration
crop_calendar["Duration_Definition"] = (
    "Representative days to first harvest/maturity"
)

# Mark these values as reference/representative values
crop_calendar["Crop_Data_Source"] = (
    "ICAR / agricultural crop production references; representative value"
)


# Save updated file
calendar_path = "/content/drive/MyDrive/Maharashtra_Vegetable_Crop_Calendar.csv"
crop_calendar.to_csv(calendar_path, index=False)


# Validation
print("Shape:", crop_calendar.shape)
print("Missing crop durations:",
      crop_calendar["Crop_Duration_Days"].isna().sum())

print("\nCrop Calendar:")
display(
    crop_calendar[
        [
            "Commodity",
            "Crop_Duration_Days",
            "Typical_Sowing_Season",
            "Crop_Data_Source"
        ]
    ]
)

Shape: (38, 11)
Missing crop durations: 0

Crop Calendar:


,Commodity,Crop_Duration_Days,Typical_Sowing_Season,Crop_Data_Source
0,Banana - Green,300,Year-round,ICAR / agricultural crop production references...
1,Beetroot,70,Rabi / Winter,ICAR / agricultural crop production references...
2,Bhindi(Ladies Finger),55,Summer / Kharif,ICAR / agricultural crop production references...
3,Bitter gourd,60,Summer / Kharif,ICAR / agricultural crop production references...
4,Bottle gourd,60,Summer / Kharif,ICAR / agricultural crop production references...
5,Brinjal,90,Kharif / Rabi / Summer,ICAR / agricultural crop production references...
6,Cabbage,90,Rabi / Winter,ICAR / agricultural crop production references...
7,Carrot,75,Rabi / Winter,ICAR / agricultural crop production references...
8,Cauliflower,80,Rabi / Winter,ICAR / agricultural crop production references...
9,Chilly Capsicum,90,Kharif / Rabi,ICAR / agricultural crop production references...


# MASTER DATASET

AGMARKNET — historical market price + arrivals

NASA POWER — district weather

SoilGrids — district soil

Crop calendar — crop duration + season

In [ ]:
# ============================================================
# AGRI-MARKET PROPHET
# FINAL MASTER DATASET CREATION
# ============================================================

import pandas as pd
import numpy as np

# ============================================================
# 1. LOAD DATASETS
# ============================================================

agri = pd.read_csv(
    "/content/drive/MyDrive/Maharashtra_Vegetables_2024_Jul2025.csv"
)

weather = pd.read_csv(
    "/content/drive/MyDrive/Maharashtra_District_Weather_2024_2025.csv"
)

soil = pd.read_csv(
    "/content/drive/MyDrive/Maharashtra_Soil_Master_0_30cm.csv"
)

calendar = pd.read_csv(
    "/content/drive/MyDrive/Maharashtra_Vegetable_Crop_Calendar.csv"
)

print("========== SOURCE DATASETS ==========")
print("AGMARKNET :", agri.shape)
print("Weather   :", weather.shape)
print("Soil      :", soil.shape)
print("Calendar  :", calendar.shape)


# ============================================================
# 2. STANDARDIZE DATA TYPES
# ============================================================

agri["Date"] = pd.to_datetime(agri["Date"], errors="raise")
weather["Date"] = pd.to_datetime(weather["Date"], errors="raise")

for data in [agri, weather, soil]:
    data["District"] = data["District"].astype(str).str.strip()

agri["Commodity"] = agri["Commodity"].astype(str).str.strip()
calendar["Commodity"] = calendar["Commodity"].astype(str).str.strip()


# ============================================================
# 3. CHECK KEY DUPLICATES BEFORE MERGING
# ============================================================

print("\n========== DUPLICATE CHECK ==========")

print(
    "Weather District + Date duplicates:",
    weather.duplicated(["District", "Date"]).sum()
)

print(
    "Soil District duplicates:",
    soil.duplicated(["District"]).sum()
)

print(
    "Calendar Commodity duplicates:",
    calendar.duplicated(["Commodity"]).sum()
)


# ============================================================
# 4. MERGE WEATHER
# ============================================================

master = agri.merge(
    weather,
    on=["District", "Date"],
    how="left",
    validate="many_to_one"
)

print("\nAfter Weather merge:", master.shape)


# ============================================================
# 5. MERGE SOIL
# ============================================================

master = master.merge(
    soil,
    on="District",
    how="left",
    validate="many_to_one"
)

print("After Soil merge:", master.shape)


# ============================================================
# 6. MERGE CROP CALENDAR
# ============================================================

master = master.merge(
    calendar,
    on="Commodity",
    how="left",
    validate="many_to_one"
)

print("After Calendar merge:", master.shape)


# ============================================================
# 7. REMOVE UNUSED EMPTY SUITABILITY COLUMNS
# ============================================================

unused_columns = [
    "Suitable_Temp_Min_C",
    "Suitable_Temp_Max_C",
    "Suitable_pH_Min",
    "Suitable_pH_Max",
    "Suitable_Rainfall_Min_mm",
    "Suitable_Rainfall_Max_mm"
]

# Remove only columns that actually exist
master.drop(
    columns=[col for col in unused_columns if col in master.columns],
    inplace=True
)


# ============================================================
# 8. FINAL DATASET VALIDATION
# ============================================================

print("\n========== FINAL MASTER DATASET ==========")

print("Rows :", len(master))
print("Cols :", len(master.columns))

print("\nDate range:")
print(master["Date"].min(), "to", master["Date"].max())

print("\nUnique values:")
print("Districts :", master["District"].nunique())
print("Markets   :", master["Market"].nunique())
print("Crops     :", master["Commodity"].nunique())


# ============================================================
# 9. MISSING VALUE CHECK
# ============================================================

missing = master.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)

print("\n========== MISSING VALUES ==========")

if len(missing) == 0:
    print("No missing values found. ✓")
else:
    print(missing)


# ============================================================
# 10. IMPORTANT COVERAGE CHECKS
# ============================================================

print("\n========== COVERAGE CHECK ==========")

print(
    "Rows missing weather:",
    master["Temperature_C"].isna().sum()
)

print(
    "Rows missing soil:",
    master["pH_0_30"].isna().sum()
)

print(
    "Rows missing crop duration:",
    master["Crop_Duration_Days"].isna().sum()
)

print(
    "Rows missing crop season:",
    master["Typical_Sowing_Season"].isna().sum()
)



print("\nFINAL COLUMNS")

for i, col in enumerate(master.columns, 1):
    print(f"{i:02d}. {col}")


# FINAL SAVE

master_path = (
    "/content/drive/MyDrive/"
    "Maharashtra_Agri_Master_2024_Jul2025.csv"
)

master.to_csv(master_path, index=False)

print("\n")
print("FINAL MASTER DATASET SAVED SUCCESSFULLY")
print("")
print(master_path)

print("\nSAMPLE")
display(master.head())

========== SOURCE DATASETS ==========
AGMARKNET : (287933, 10)
Weather   : (13872, 9)
Soil      : (24, 9)
Calendar  : (38, 11)

========== DUPLICATE CHECK ==========
Weather District + Date duplicates: 0
Soil District duplicates: 0
Calendar Commodity duplicates: 0

After Weather merge: (287933, 17)
After Soil merge: (287933, 25)
After Calendar merge: (287933, 35)

========== FINAL MASTER DATASET ==========
Rows : 287933
Cols : 29

Date range:
2024-01-01 00:00:00 to 2025-07-31 00:00:00

Unique values:
Districts : 24
Markets   : 129
Crops     : 38

========== MISSING VALUES ==========
No missing values found. ✓

========== COVERAGE CHECK ==========
Rows missing weather: 0
Rows missing soil: 0
Rows missing crop duration: 0
Rows missing crop season: 0

========== FINAL COLUMNS ==========
01. State
02. District
03. Market
04. Commodity Group
05. Commodity
06. Date
07. Arrival Quantity
08. Arrival Unit
09. Modal Price
10. Price Unit
11. Latitude_x
12. Longitude_x
13. Temperature_C
14. Min_Te

,State,District,Market,Commodity Group,Commodity,Date,Arrival Quantity,Arrival Unit,Modal Price,Price Unit,...,pH_0_30,Organic_Carbon_g_kg_0_30,Nitrogen_g_kg_0_30,Clay_percent_0_30,CEC_cmolc_kg_0_30,Soil_Data_Method,Crop_Duration_Days,Typical_Sowing_Season,Crop_Data_Source,Duration_Definition
0,Maharashtra,Ahilyanagar,Ahilyanagar APMC,Vegetables,Onion,2024-11-04,4729.3,Metric Tonnes,3450.0,Rs./Quintal,...,7.119,9.091,1.063,42.19,31.423,SoilGrids,120,Kharif / Rabi,ICAR / agricultural crop production references...,Representative days to first harvest/maturity
1,Maharashtra,Ahilyanagar,Ahilyanagar APMC,Vegetables,Bhindi(Ladies Finger),2024-11-05,3.3,Metric Tonnes,2750.0,Rs./Quintal,...,7.119,9.091,1.063,42.19,31.423,SoilGrids,55,Summer / Kharif,ICAR / agricultural crop production references...,Representative days to first harvest/maturity
2,Maharashtra,Ahilyanagar,Ahilyanagar APMC,Vegetables,Bitter gourd,2024-11-05,3.1,Metric Tonnes,2500.0,Rs./Quintal,...,7.119,9.091,1.063,42.19,31.423,SoilGrids,60,Summer / Kharif,ICAR / agricultural crop production references...,Representative days to first harvest/maturity
3,Maharashtra,Ahilyanagar,Ahilyanagar APMC,Vegetables,Bottle gourd,2024-11-05,3.4,Metric Tonnes,1700.0,Rs./Quintal,...,7.119,9.091,1.063,42.19,31.423,SoilGrids,60,Summer / Kharif,ICAR / agricultural crop production references...,Representative days to first harvest/maturity
4,Maharashtra,Ahilyanagar,Ahilyanagar APMC,Vegetables,Brinjal,2024-11-05,5.1,Metric Tonnes,3250.0,Rs./Quintal,...,7.119,9.091,1.063,42.19,31.423,SoilGrids,90,Kharif / Rabi / Summer,ICAR / agricultural crop production references...,Representative days to first harvest/maturity


In [ ]:
master.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 287933 entries, 0 to 287932
Data columns (total 29 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   State                      287933 non-null  object        
 1   District                   287933 non-null  object        
 2   Market                     287933 non-null  object        
 3   Commodity Group            287933 non-null  object        
 4   Commodity                  287933 non-null  object        
 5   Date                       287933 non-null  datetime64[ns]
 6   Arrival Quantity           287933 non-null  float64       
 7   Arrival Unit               287933 non-null  object        
 8   Modal Price                287933 non-null  float64       
 9   Price Unit                 287933 non-null  object        
 10  Latitude_x                 287933 non-null  float64       
 11  Longitude_x                287933 non-null  float64 